In [1]:
import numpy as np
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath('../../..'))
import matplotlib.pyplot as plt
from scripts import nodes as n
from scripts import elements as e
from scripts import material_params as mat
from scipy.linalg import eigh
import plotly.graph_objects as go

In [2]:
nodes = []
nodal_values = np.loadtxt('../../../text_files/nodes_minimal_model_.txt', delimiter=',')
for i in range(nodal_values.shape[0]):
  nodes.append(n.nodes(nodal_values[i, 0], nodal_values[i, 1], nodal_values[i, 2]))

# All fixed parameters

In [3]:
# Truss parameters
E = 210e9  # Young's modulus in Pascals
nu = 0.3   # Poisson's ratio
G = E / (2 * (1 + nu))  # Shear modulus in Pascals
m = 6.7504 * 10 ** 6 #kg
d1 = 0.8 #m
d2 = 1.8 #m
t1 = 0.03 #m
t2 = 0.08 #m
L_truss_element_y = 15 #m
h_truss = 18 #m
Aeq, Ieqy, Ieqz, b_eq, h_eq = mat.effective_truss_stiffness(d1, d2, t1, t2, h_truss, L_truss_element_y) 
L = 237.5
rho_truss = m / (L * Aeq)
It = (b_eq * h_eq**3 / 3) * (1 - 0.63 * (h_eq / b_eq) * (1 - (h_eq**4 / (12 * b_eq**4)))) *0.06
k = 0.08
Ip  = Ieqy + Ieqz
ep_K = [E, G, Aeq, Ieqy, Ieqz, It, k]
ep_m = [rho_truss, Aeq, Ieqy, Ieqz, Ip]

k = 5/6
rho = 7850 #kg/m^3
E=210e9 #Pa
G = E/(2*(1+nu)) #Pa
nu = 0.3 #Poisson's ratio
k_fender = mat.stiffness_fenders()
Iy_connect, Iz_connect, Ip_connect, It_connect, A_connect = mat.stiffness_connecting_beams()
ep_K_connect = [E, G, A_connect, Iy_connect, Iz_connect, It_connect, k]
ep_m_connect = [rho, A_connect, Iy_connect, Iz_connect, Ip_connect]

# Sensitivity analysis

In [6]:
N = 50

# Ball-joint parameters
kr_min = 0
kr_max = 1e7
kr = np.arange(kr_min, kr_max, (kr_max - kr_min) / N)

# Fender parameters
kf_min = 1e8
kf_max = 1e13
kf = np.arange(kf_min, kf_max, (kf_max - kf_min) / N)


# Retaining wall parameters
Iy_min = 1500
Iy_max = 10000
Iz_max = 0.5 * Iy_max #m^4 
It_max = 0.05 * Iy_max #m^4
Ip_max = Iy_max + Iz_max #m^4
Iy_min = 1500 #m^4
Iz_min = 0.5 * Iy_min #m^4
It_min = 0.05 * Iy_min #m^4
Ip_min = Iy_min + Iz_min #m^4
h = 22 #m
A_eq_max = Iy_max * (12 / h**2) #m^2
A_eq_min = Iy_min * (12 / h**2) #m^2
L = 237.5 #m
m = 6455148 #kg

Iy = np.arange(Iy_min, Iy_max, (Iy_max - Iy_min) / N)
Iz = np.arange(Iz_min, Iz_max, (Iz_max - Iz_min) / N)
It = np.arange(It_min, It_max, (It_max - It_min) / N)
Ip = np.arange(Ip_min, Ip_max, (Ip_max - Ip_min) / N)
A_eq = np.arange(A_eq_min, A_eq_max, (A_eq_max - A_eq_min) / N)
rho = np.zeros(N)
for i in range(N):
    rho[i] = m / (L * A_eq[i])

k = 1 #Shear factor for hollow closed sections



In [7]:
param_space = np.zeros((N, N, N, 10))
NN = len(nodes)
DOFS_per_node = 6
fender_dofs = [70, 75, 79, 83, 87, 91, 95, 99, 103, 107, 111, 115, 119, 123, 128]
dofs = n.degrees_of_freedom(nodes)
kr_dofs = [dofs['dof_1'][3], dofs['dof_1'][4], dofs['dof_1'][5]] 

for i in range(len(Iy)):
    ep_K_wall = [E, G, A_eq[i], Iy[i], Iz[i], It[i], k]
    ep_m_wall = [rho[i], A_eq[i], Iy[i], Iz[i], Ip[i]]

    elements = []
    element_nodes = np.loadtxt('../../../text_files/element_nodes3.txt', dtype=int)

    for elem in range(element_nodes.shape[0]):
        if element_nodes[elem, 2] == 0:
            elements.append(e.elements(nodes[element_nodes[elem, 0] - 1], nodes[element_nodes[elem, 1] - 1], ep_K, ep_m))
        elif element_nodes[elem, 2] == 1:
            elements.append(e.elements(nodes[element_nodes[elem, 0] - 1], nodes[element_nodes[elem, 1] - 1], ep_K_wall, ep_m_wall))
        elif element_nodes[elem, 2] == 2:
            elements.append(e.elements(nodes[element_nodes[elem, 0] - 1], nodes[element_nodes[elem, 1] - 1], ep_K_connect, ep_m_connect))

    element_nodes = np.loadtxt('../../../text_files/element_nodes.txt', dtype=int)
    element_locs = []

    for (nA, nB) in element_nodes:
        dofs_A = dofs[f'dof_{nA}']
        dofs_B = dofs[f'dof_{nB}']
        element_locs.append(np.hstack((dofs_A, dofs_B)))
    


    for j in range(len(kf)):


        K_global = np.zeros((NN * DOFS_per_node, NN * DOFS_per_node))
        M_global = np.zeros((NN * DOFS_per_node, NN * DOFS_per_node))

        K_locs = []
        M_locs = []

        for elem_loc in range(len(element_locs)):
            K_global[np.ix_(element_locs[elem_loc], element_locs[elem_loc])] += elements[elem_loc][-1]
            M_global[np.ix_(element_locs[elem_loc], element_locs[elem_loc])] += elements[elem_loc][-2]

        K_global = 0.5 * (K_global + K_global.T)
        M_global = 0.5 * (M_global + M_global.T)

        k_fender_dofs = [dofs[f'dof_{i+1}'][2] for i in fender_dofs]
        for dof in k_fender_dofs:
            K_global[dof, dof] += kf[j]


        for k in range(len(kr)):

            for dof in kr_dofs:
                K_global[dof, dof] += kr[k]

            indices_to_remove = np.hstack((dofs['dof_1'][0:3], dofs['dof_129'][0:2]))
            keep_indices = np.setdiff1d(np.arange(NN * DOFS_per_node), indices_to_remove)
            K_global_reduced = K_global[np.ix_(keep_indices, keep_indices)]
            M_global_reduced = M_global[np.ix_(keep_indices, keep_indices)]
            eigvals_global, eigvecs_global = eigh(K_global_reduced, M_global_reduced)   

            tol = 1e-6
            positive = eigvals_global > tol
            eigvals_global = eigvals_global[positive]
            eigvecs_global = eigvecs_global[:, positive]

            frequencies_rad = np.sqrt(eigvals_global)
            frequencies_hz = frequencies_rad / (2 * np.pi)



            K_global = np.zeros((NN * DOFS_per_node, NN * DOFS_per_node))
            M_global = np.zeros((NN * DOFS_per_node, NN * DOFS_per_node))

            K_locs = []
            M_locs = []

            for elem_loc in range(len(element_locs)):
                K_global[np.ix_(element_locs[elem_loc], element_locs[elem_loc])] += elements[elem_loc][-1]
                M_global[np.ix_(element_locs[elem_loc], element_locs[elem_loc])] += elements[elem_loc][-2]

            K_global = 0.5 * (K_global + K_global.T)
            M_global = 0.5 * (M_global + M_global.T)

            k_fender_dofs = [dofs[f'dof_{i+1}'][2] for i in fender_dofs]
            for dof in k_fender_dofs:
                K_global[dof, dof] += kf[j]

            param_space[i, j, k, :] = frequencies_hz[:10]

    print(f'Completed iteration {i+1} out of {len(Iy)}')

    
file_name = f'param_space_N{N}_kr0_{kr_max}_Iy{int(Iy_min)}_{int(Iy_max)}.npy'
np.save(f'3param_runs/{file_name}', param_space)

Completed iteration 1 out of 50
Completed iteration 2 out of 50
Completed iteration 3 out of 50
Completed iteration 4 out of 50
Completed iteration 5 out of 50
Completed iteration 6 out of 50
Completed iteration 7 out of 50
Completed iteration 8 out of 50
Completed iteration 9 out of 50
Completed iteration 10 out of 50
Completed iteration 11 out of 50
Completed iteration 12 out of 50
Completed iteration 13 out of 50
Completed iteration 14 out of 50
Completed iteration 15 out of 50
Completed iteration 16 out of 50
Completed iteration 17 out of 50
Completed iteration 18 out of 50
Completed iteration 19 out of 50
Completed iteration 20 out of 50
Completed iteration 21 out of 50
Completed iteration 22 out of 50
Completed iteration 23 out of 50
Completed iteration 24 out of 50
Completed iteration 25 out of 50
Completed iteration 26 out of 50
Completed iteration 27 out of 50
Completed iteration 28 out of 50
Completed iteration 29 out of 50
Completed iteration 30 out of 50
Completed iteration

In [ ]:
import numpy as np
import plotly.graph_objects as go

# param_space has shape (N, N, N, 10)
N = param_space.shape[0]

# --- real parameter ranges ---
kr = np.arange(kr_min, kr_max, (kr_max - kr_min) / N)
kf = np.arange(kf_min, kf_max, (kf_max - kf_min) / N)
Iy = np.arange(Iy_min, Iy_max, (Iy_max - Iy_min) / N)

# choose natural frequency to plot (0–9)
freq_id = 0

# 3D volume for that frequency
volume = param_space[..., freq_id]  # shape (N, N, N)

# generate full coordinate grid in REAL VALUES
Iy_grid, kf_grid, kr_grid = np.meshgrid(Iy, kf, kr, indexing="ij")

# flatten everything for 3D scatter
x = Iy_grid.flatten()
y = kf_grid.flatten()
z = kr_grid.flatten()
c = volume.flatten()

# ---- INTERACTIVE PLOT ----
fig = go.Figure(data=[go.Scatter3d(
    x=x,
    y=y,
    z=z,
    mode='markers',
    marker=dict(
        size=3,
        color=c,             # color by natural frequency
        colorscale='Viridis',
        colorbar=dict(title=f"Natural frequency {freq_id+1}"),
        opacity=0.8
    )
)])

fig.update_layout(
    scene=dict(
        xaxis_title='Iy',
        yaxis_title='kf',
        zaxis_title='kr'
    ),
    title="3D Parameter Space (interactive)",
    width=900,
    height=700
)

fig.show()